# Resume text → structured JSON: QLoRA fine-tune of Qwen2.5-1.5B-Instruct

In [1]:
!pip install -q -U bitsandbytes transformers peft accelerate datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 73.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 23.9 MB/s eta 0:00:00


In [2]:
import os, re, json, random, time, gc, glob
import numpy as np, pandas as pd, torch
from datasets import load_dataset, Dataset
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
                          TrainingArguments, Trainer, DataCollatorForSeq2Seq, set_seed)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

os.environ["WANDB_DISABLED"] = "true"
SEED = 42
set_seed(SEED)

MODEL   = "Qwen/Qwen2.5-1.5B-Instruct"
SMOKE = False     # True = quick pipeline check; set False for the full run
N_REAL = 40
N_TRAIN, N_VAL, N_TEST = (64, 20, 20) if SMOKE else (1000, 100, 100)   # test set is rendered twice (seen + unseen format)
MAX_LEN = 1536      # max tokens (prompt + answer) per example
MAX_NEW = 768       # max generated tokens at eval (high enough not to cut off the verbose zero-shot baseline)
TRAIN_STYLES, UNSEEN_STYLE = [0, 1], 2

# Where results/checkpoints go: Google Drive on Colab, /kaggle/working on Kaggle, else current folder
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/resume-qlora/full"
except ImportError:
    OUT_DIR = "/kaggle/working" if os.path.exists("/kaggle") else "."
os.makedirs(OUT_DIR, exist_ok=True)
print("SMOKE =", SMOKE, "| OUT_DIR =", OUT_DIR)

Mounted at /content/drive
SMOKE = False | OUT_DIR = /content/drive/MyDrive/resume-qlora/full


### Load dataset (falls back to raw JSON files if Arrow schema merging fails) 

In [ ]:
try:
    ds = load_dataset("datasetmaster/resumes")
    split = list(ds.keys())[0]
    records = [dict(r) for r in ds[split]]
except Exception as e:
    print("load_dataset failed, falling back to raw files:", e)
    from huggingface_hub import snapshot_download
    p = snapshot_download("datasetmaster/resumes", repo_type="dataset")
    files = glob.glob(p + "/**/*.json", recursive=True) + glob.glob(p + "/**/*.jsonl", recursive=True)
    print(files)
    records = []
    for f in files:
        txt = open(f).read()
        try:
            obj = json.loads(txt)
            records += obj if isinstance(obj, list) else [obj]
        except json.JSONDecodeError:
            records += [json.loads(l) for l in txt.splitlines() if l.strip()]

print(len(records), "records")
print(json.dumps(records[0], indent=2, default=str)[:3000])

README.md:   0%|          | 0.00/5.95k [00:00<?, ?B/s]

master_resumes.jsonl: reconstructing file:   0%|          |  0.00B / 16.3MB            

master_resumes.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/4817 [00:00<?, ? examples/s]

4817 records
{
  "personal_info": {
    "name": "Unknown",
    "email": "Unknown",
    "phone": "Unknown",
    "location": {
      "city": "Pune",
      "country": "India",
      "remote_preference": "Unknown"
    },
    "summary": "Python Developer with experience in Python, Tensorflow, Numpy, C, C++, MySQL, and various platforms.",
    "linkedin": "Unknown",
    "github": "Unknown"
  },
  "experience": [
    {
      "company": "Fresher",
      "company_info": {
        "industry": "Unknown",
        "size": "Unknown"
      },
      "title": "Python Developer",
      "level": "entry",
      "employment_type": "full-time",
      "dates": {
        "start": "Unknown",
        "end": "Present",
        "duration": "6 months"
      },
      "responsibilities": [
        "Python programming"
      ],
      "technical_environment": {
        "technologies": [
          "Python",
          "Tensorflow",
          "Numpy",
          "C",
          "C++",
          "MySQL"
        ],
        "

### Cleaning + safe access helpers 

In [ ]:

PLACEHOLDERS = {"unknown", "n/a", "na", "none", "null", "not specified", "not available", "-"}

def nk(k):
    """Normalize a key: 'Personal_Info' -> 'personalinfo'."""
    return re.sub(r"[^a-z0-9]", "", str(k).lower())

def empty(x):
    """True for None, empty containers, and anonymization placeholders like 'Unknown'."""
    if isinstance(x, str) and x.strip().lower() in PLACEHOLDERS:
        return True
    return x is None or x == "" or x == [] or x == {}

def is_scalar(x):
  return isinstance(x, (str, int, float, bool))

def clip(x, n=300):
  return str(x).strip()[:n]

def as_list(x):
  return [] if x is None else (x if isinstance(x, list) else [x])

def get(d, *cands):
    """First non-empty value whose normalized key equals, then contains, a candidate."""
    if not isinstance(d, dict):
      return None

    keys = {nk(k): k for k in d}

    for c in cands:                                   # exact match first
        if c in keys and not empty(d[keys[c]]): return d[keys[c]]

    for c in cands:                                   # then substring match
        for k, orig in keys.items():
            if c in k and not empty(d[orig]): return d[orig]

    return None

def flat_strings(x):
    """All non-empty strings inside any nested structure, in order."""
    if empty(x):
      return []

    if is_scalar(x):
      return [clip(x, 100)]

    if isinstance(x, list):
      return [s for i in x for s in flat_strings(i)]

    if isinstance(x, dict):
      return [s for v in x.values() for s in flat_strings(v)]

    return []

def text_of(x):
    """Best single string for a value that may be a scalar, dict (e.g. {'name': ...}) or list."""
    if empty(x):
      return ""

    if is_scalar(x):
      return str(x).strip()

    if isinstance(x, dict):
       return text_of(get(x, "name", "title"))   # dict with no usable name -> ""

    return " ".join(flat_strings(x)[:3])

def dedupe(seq):
    seen, out = set(), []
    for s in seq:
        if s.lower() not in seen:
            seen.add(s.lower()); out.append(s)
    return out

### Target extraction: raw record -> the trimmed JSON the model must produce

In [ ]:

MAX_EDU, MAX_EXP, MAX_SKILLS = 5, 4, 25     # caps keep the answer short (fewer tokens)

def degree_parts(e):
    # degree may be a dict like {"level": "BSc", "field": "Computer Science", "major": ...} or a plain string
    d = get(e, "degree", "qualification")
    if isinstance(d, dict):
        return text_of(get(d, "level", "type")), text_of(get(d, "field", "specialization", "discipline"))
    return text_of(d), text_of(get(e, "fieldofstudy", "field"))

def extract_education(rec):
    out = []
    for e in as_list(get(rec, "education"))[:MAX_EDU]:
        if isinstance(e, dict):
            deg, fld = degree_parts(e)
            out.append({"institution": text_of(get(e, "institution", "school", "university", "college")),
                        "degree": deg, "field": fld})
    return out

def extract_experience(rec):
    out = []
    for e in as_list(get(rec, "experience"))[:MAX_EXP]:
        if isinstance(e, dict):
            out.append({"company": text_of(get(e, "company", "employer", "organization")),
                        "title":   text_of(get(e, "jobtitle", "title", "position", "role"))})
    return out

SPOKEN_KEYS = {"languages", "spokenlanguages", "language"}   # spoken languages are not technical skills

def skill_names(x):
    # {"name": "Python", "level": "intermediate"} -> "Python" (level is dropped)
    if empty(x): return []
    if is_scalar(x): return [clip(x, 100)]
    if isinstance(x, list): return [n for i in x for n in skill_names(i)]
    if isinstance(x, dict):
        if "name" in {nk(k) for k in x}: return skill_names(get(x, "name"))
        return [n for k, v in x.items() if nk(k) not in SPOKEN_KEYS for n in skill_names(v)]
    return []

def extract_skills(rec):
    return dedupe(skill_names(get(rec, "skill")))[:MAX_SKILLS]

def make_target(rec):
    personal = get(rec, "personal") or {}
    return {"name":       text_of(get(personal, "fullname", "name")),
            "email":      text_of(get(personal, "email")),
            "skills":     extract_skills(rec),
            "education":  extract_education(rec),
            "experience": extract_experience(rec)}

### Layout renderers: nested JSON -> text (each style is a different "resume look")

In [ ]:

def _lab(k): return str(k).replace("_", " ").strip().title()

def r_labeled(v, ind, bold, bullet):
    """'Label: value' lines. Style 0 uses plain labels and '-'; style 2 uses **bold** labels and '*'."""
    pad, out = "  " * ind, []
    if isinstance(v, dict):
        for k, x in v.items():
            if empty(x): continue
            lab = f"**{_lab(k)}**" if bold else _lab(k)
            if is_scalar(x):
                out.append(f"{pad}{lab}: {clip(x)}")
            else:
                out.append(f"{pad}{lab}:"); out += r_labeled(x, ind + 1, bold, bullet)
    elif isinstance(v, list):
        cap = 40 if all(is_scalar(i) for i in v) else 6      # long scalar lists (skills) vs entry lists
        for x in v[:cap]:
            if empty(x): continue
            if is_scalar(x):
                out.append(f"{pad}{bullet} {clip(x)}")
            else:
                out += r_labeled(x, ind + 1, bold, bullet); out.append("")
    elif not empty(v):
        out.append(f"{pad}{clip(v)}")
    return out

def r_values(v, ind):
    """Style 1: labels dropped, values joined with ' | ' (the model must infer which is which)."""
    pad, out = "  " * ind, []
    if isinstance(v, dict):
        sc = [clip(x, 200) for x in v.values() if is_scalar(x) and not empty(x)]
        if sc: out.append(pad + " | ".join(sc))
        for x in v.values():
            if not is_scalar(x) and not empty(x): out += r_values(x, ind + 1)
    elif isinstance(v, list):
        if all(is_scalar(i) for i in v):
            out.append(pad + ", ".join(clip(i, 100) for i in v[:40] if not empty(i)))
        else:
            for x in v[:6]:
                if empty(x): continue
                out += r_values(x, ind); out.append("")
    elif not empty(v):
        out.append(pad + clip(v))
    return out

### Full resume renderer

style 0: "=== SECTION ===" + Label: value | style 1: SECTION + values only | style 2: "## Section" + **Label**: value

In [ ]:

def render_resume(rec, style, seed):
    rng = random.Random(seed)
    secs = [(k, v) for k, v in rec.items() if not empty(v)]
    pers = [s for s in secs if "personal" in nk(s[0])]          # personal info stays on top
    rest = [s for s in secs if "personal" not in nk(s[0])]
    rng.shuffle(rest)                                            # other sections in random order
    out = []
    for k, v in pers + rest:
        lab = _lab(k)
        if style == 0:
            out.append(f"=== {lab.upper()} ==="); out += r_labeled(v, 0, False, "-")
        elif style == 1:
            out.append(lab.upper());              out += r_values(v, 0)
        else:
            out.append(f"## {lab}");              out += r_labeled(v, 0, True, "*")
        out.append("")
    return "\n".join(out).strip()

# Quick data check 

In [ ]:

from collections import Counter
sample = [make_target(r) for r in records[:500]]
print("sample target:", json.dumps(sample[0], ensure_ascii=False)[:500])
print("\ntop companies:", Counter(e["company"] for t in sample for e in t["experience"]).most_common(8))
print("top titles   :", Counter(e["title"] for t in sample for e in t["experience"]).most_common(8))
print(f"\nout of {len(sample)}: no skills={sum(not t['skills'] for t in sample)}, "
      f"no experience={sum(not t['experience'] for t in sample)}, "
      f"no education={sum(not t['education'] for t in sample)}, "
      f"real name={sum(bool(t['name']) for t in sample)}, real email={sum(bool(t['email']) for t in sample)}")

sample target: {"name": "", "email": "", "skills": ["Python", "C++", "C", "Tensorflow", "Numpy", "MySQL"], "education": [{"institution": "Savitribai Phule Pune University", "degree": "ME", "field": "Computer Engineering"}, {"institution": "Savitribai Phule Pune University", "degree": "B.E", "field": "Computer Engineering"}], "experience": [{"company": "Fresher", "title": "Python Developer"}]}

top companies: [('Parkar Consulting and Labs', 15), ('Nendrasys Technologies Pvt Ltd', 14), ('KPIT Technologies', 14), ('Tech Mahindra', 13), ('Salcluster Technologies', 12), ('Replete Business Solutions Pvt Ltd', 10), ('Delta Controls, Dubai FZCO', 8), ('Honeywell Automation India Ltd', 8)]
top titles   : [('Java Developer', 290), ('Python Developer', 229), ('Javascript Developer', 169), ('DevOps Engineer', 15), ('Devops Engineer', 14), ('Business Analyst', 14), ('Python Developer/analyst', 13), ('Operations Manager', 8)]

out of 500: no skills=14, no experience=0, no education=4, real name=335,

# Prompt, splits, tokenization

In [ ]:

SYSTEM = ("You extract structured data from resumes. Return ONLY a valid JSON object with exactly this schema, no extra text:\n"
          '{"name": str, "email": str, "skills": [str], '
          '"education": [{"institution": str, "degree": str, "field": str}], '
          '"experience": [{"company": str, "title": str}]}\n'
          "skills = technical skills only (no proficiency levels, no spoken languages). "
          "degree = the degree level (e.g. BSc, MSc, B.Tech); field = the subject studied. "
          "Use an empty string or empty list if a field is missing.")

tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None: tok.pad_token = tok.eos_token
IM_END = tok.convert_tokens_to_ids("<|im_end|>")

def chat_prompt(resume_text):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": resume_text}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

targets = {i: make_target(r) for i, r in enumerate(records)}
usable = [i for i, t in targets.items() if len(t["skills"]) > 0 and (t["experience"] or t["education"])]
print(f"{len(usable)}/{len(records)} records usable (>=1 skill and some experience/education)")
print("of those, with a real name:", sum(bool(targets[i]["name"]) for i in usable),
      "| with a real email:", sum(bool(targets[i]["email"]) for i in usable))

real_ids = [i for i in usable if not targets[i]["name"]]   # anonymized real resumes have no name
syn_ids  = [i for i in usable if targets[i]["name"]]
random.Random(SEED).shuffle(real_ids)
test_real_ids = real_ids[:N_REAL]                          # held out of training entirely
ids = syn_ids + real_ids[N_REAL:]
random.Random(SEED).shuffle(ids)

n_train = min(N_TRAIN, len(ids) - N_VAL - N_TEST)
train_ids = ids[:n_train]
val_ids   = ids[n_train:n_train + N_VAL]
test_ids  = ids[n_train + N_VAL:n_train + N_VAL + N_TEST]   # split by record, so no resume appears in two splits

def build(id_list, styles):
    return [{"id": i, "style": styles[j % len(styles)], "target": targets[i],
             "text": render_resume(records[i], styles[j % len(styles)], SEED + i)}
            for j, i in enumerate(id_list)]

def tokenize(ex):
    p = tok(chat_prompt(ex["text"]), add_special_tokens=False)["input_ids"]
    a = tok(json.dumps(ex["target"], ensure_ascii=False) + "<|im_end|>", add_special_tokens=False)["input_ids"]
    return {"input_ids": p + a, "attention_mask": [1] * (len(p) + len(a)),
            "labels": [-100] * len(p) + a, "n_prompt": len(p)}   # loss only on the JSON answer

def tokenize_all(exs):
    toks = [tokenize(e) for e in exs]
    keep = [(e, t) for e, t in zip(exs, toks) if len(t["input_ids"]) <= MAX_LEN]
    print(f"kept {len(keep)}/{len(exs)} (<= {MAX_LEN} tokens); "
          f"median len {int(np.median([len(t['input_ids']) for t in toks]))}, "
          f"p95 {int(np.percentile([len(t['input_ids']) for t in toks], 95))}")
    return [e for e, _ in keep], [t for _, t in keep]

train_ex, train_tok = tokenize_all(build(train_ids, TRAIN_STYLES))
val_ex,   val_tok   = tokenize_all(build(val_ids,   TRAIN_STYLES))
test_seen,   _      = tokenize_all(build(test_ids,  TRAIN_STYLES))
test_unseen, _      = tokenize_all(build(test_ids,  [UNSEEN_STYLE]))
test_real, _ = tokenize_all(build(test_real_ids, TRAIN_STYLES))
print("held-out real records:", len(test_real))

strip = lambda ts: [{k: t[k] for k in ("input_ids", "attention_mask", "labels")} for t in ts]
train_ds, val_ds = Dataset.from_list(strip(train_tok)), Dataset.from_list(strip(val_tok))

# ---- SANITY CHECK: read these before training ----
for ex in (train_ex[0], test_unseen[0]):
    print("=" * 80, f"\nstyle {ex['style']} | INPUT TEXT (first 900 chars):\n{ex['text'][:900]}")
    print("\nTARGET:", json.dumps(ex["target"], ensure_ascii=False)[:600])

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

4803/4817 records usable (>=1 skill and some experience/education)
of those, with a real name: 4650 | with a real email: 4640
kept 1000/1000 (<= 1536 tokens); median len 896, p95 1300
kept 100/100 (<= 1536 tokens); median len 882, p95 1303
kept 100/100 (<= 1536 tokens); median len 885, p95 1282
kept 98/100 (<= 1536 tokens); median len 1150, p95 1462
kept 40/40 (<= 1536 tokens); median len 608, p95 1121
held-out real records: 40
style 0 | INPUT TEXT (first 900 chars):
=== PERSONAL INFO ===
Name: Michael Myers
Email: wmay@example.net
Phone: +1-793-222-8985x81143
Location:
  City: Davisshire
  Country: Burundi
  Remote Preference: remote
Summary: Future require raise remain center arrive. Long least along game purpose situation debate maintain. Cut week democratic affect minute. Interview soldier or movement kitchen purpose.
Linkedin: linkedin.com/in/johnsonmelanie
Github: github.com/smolina

=== SKILLS ===
Technical:
  Programming Languages:
      Name: C++
      Level: beginner

      N

In [10]:
def target_strings(t):
    s = [t["name"], t["email"]] + t["skills"]
    s += [v for e in t["education"] for v in e.values()]
    s += [v for e in t["experience"] for v in e.values()]
    return [x for x in s if x]

tot = hit = 0; misses = []
for ex in train_ex + val_ex + test_seen + test_unseen:
    for x in target_strings(ex["target"]):
        tot += 1
        if x in ex["text"]: hit += 1
        else: misses.append((ex["id"], ex["style"], x))
print(f"target strings found verbatim in input text: {hit}/{tot} = {hit/tot:.1%}")
print(misses[:10])

target strings found verbatim in input text: 22882/22888 = 100.0%
[(181, 0, 'Jquery'), (181, 0, 'JavaEE'), (181, 0, 'Mysql'), (83, 0, 'Keras'), (101, 0, 'Keras'), (101, 2, 'Keras')]


# Metrics + generation

In [ ]:

def norm(s): return re.sub(r"\s+", " ", re.sub(r"[^\w\s@.+-]", "", str(s).lower())).strip()

def parse_json(txt):
    txt = re.sub(r"^```(?:json)?|```$", "", txt.strip(), flags=re.M).strip()
    s, e = txt.find("{"), txt.rfind("}")
    if s == -1 or e <= s: return None
    try:
        o = json.loads(txt[s:e + 1]); return o if isinstance(o, dict) else None
    except Exception:
        return None

def schema_ok(o):
    if o is None or set(o.keys()) != {"name", "email", "skills", "education", "experience"}: return False
    if not (isinstance(o["name"], str) and isinstance(o["email"], str)): return False
    if not (isinstance(o["skills"], list) and all(isinstance(s, str) for s in o["skills"])): return False
    for key, fields in (("education", ("institution", "degree", "field")), ("experience", ("company", "title"))):
        if not isinstance(o[key], list): return False
        for it in o[key]:
            if not (isinstance(it, dict) and set(it.keys()) == set(fields)
                    and all(isinstance(it[f], str) for f in fields)): return False
    return True

def f1(pred, gold):
    if not pred and not gold: return 1.0
    if not pred or not gold: return 0.0
    tp = len(pred & gold)
    if tp == 0: return 0.0
    p, r = tp / len(pred), tp / len(gold)
    return 2 * p * r / (p + r)

def score(o, gold):
    zero = {"name": 0.0 if gold["name"] else np.nan, "email": 0.0 if gold["email"] else np.nan,
            "skills": 0.0, "education": 0.0, "experience": 0.0}
    if o is None: return zero          # unparseable output scores 0 on every field
    try:
        tup = lambda lst, keys: {tuple(norm(i.get(k, "")) for k in keys) for i in lst if isinstance(i, dict)}
        return {"name": float(norm(o.get("name", "")) == norm(gold["name"])) if gold["name"] else np.nan,
                "email": float(norm(o.get("email", "")) == norm(gold["email"])) if gold["email"] else np.nan,
                "skills": f1({norm(s) for s in o.get("skills", []) if isinstance(s, str)},
                             {norm(s) for s in gold["skills"]}),
                "education": f1(tup(o.get("education", []), ("institution", "degree", "field")),
                                tup(gold["education"], ("institution", "degree", "field"))),
                "experience": f1(tup(o.get("experience", []), ("company", "title")),
                                 tup(gold["experience"], ("company", "title")))}
    except Exception:
        return zero

@torch.no_grad()
def generate_batch(model, prompts, bs=8):
    tok.padding_side = "left"
    model.eval(); outs = []
    for i in range(0, len(prompts), bs):
        b = tok(prompts[i:i + bs], return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        g = model.generate(**b, max_new_tokens=MAX_NEW, do_sample=False,
                           pad_token_id=tok.pad_token_id, eos_token_id=IM_END)
        outs += tok.batch_decode(g[:, b["input_ids"].shape[1]:], skip_special_tokens=True)
    return outs

def evaluate(model, examples, name):
    prompts = [chat_prompt(e["text"]) for e in examples]
    t0 = time.time(); raw = generate_batch(model, prompts); dt = time.time() - t0
    rows = []
    for e, r in zip(examples, raw):
        o = parse_json(r); s = score(o, e["target"])
        rows.append({"id": e["id"], "raw": r, "json_valid": o is not None, "schema_valid": schema_ok(o),
                     **{f"f_{k}": v for k, v in s.items()}})
    df = pd.DataFrame(rows)
    fcols = [c for c in df.columns if c.startswith("f_")]
    df["field_mean"] = df[fcols].mean(axis=1)
    summ = {"set": name, "n": len(df), "json_valid": df.json_valid.mean(), "schema_valid": df.schema_valid.mean(),
            **{c: df[c].mean() for c in fcols}, "field_mean": df.field_mean.mean(),
            "sec_per_sample": dt / len(df)}
    return df, summ

# Load base model in 4-bit + ZERO-SHOT BASELINE

In [ ]:

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb,
                                             device_map={"": 0}, torch_dtype=torch.float16)
model.config.use_cache = True

dfs, summaries = {}, []
for tag, exs in (("seen formats", test_seen), ("unseen format", test_unseen), ("real records", test_real)):
    dfs[f"base | {tag}"], s = evaluate(model, exs, f"zero-shot | {tag}")
    summaries.append(s); print(s)
json.dump(summaries, open(f"{OUT_DIR}/baseline_results.json", "w"), indent=2)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

{'set': 'zero-shot | seen formats', 'n': 100, 'json_valid': np.float64(0.96), 'schema_valid': np.float64(0.2), 'f_name': np.float64(0.9072164948453608), 'f_email': np.float64(0.9270833333333334), 'f_skills': np.float64(0.46330596845534927), 'f_education': np.float64(0.56), 'f_experience': np.float64(0.705), 'field_mean': np.float64(0.7084560654859415), 'sec_per_sample': 6.602554523944855}
{'set': 'zero-shot | unseen format', 'n': 98, 'json_valid': np.float64(0.8571428571428571), 'schema_valid': np.float64(0.01020408163265306), 'f_name': np.float64(0.8526315789473684), 'f_email': np.float64(0.8404255319148937), 'f_skills': np.float64(0.5444156883532633), 'f_education': np.float64(0.47108843537414963), 'f_experience': np.float64(0.7959183673469388), 'field_mean': np.float64(0.7004899435631728), 'sec_per_sample': 6.507263061951618}
{'set': 'zero-shot | real records', 'n': 40, 'json_valid': np.float64(1.0), 'schema_valid': np.float64(0.55), 'f_name': nan, 'f_email': nan, 'f_skills': np.flo

In [ ]:
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                           bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb,
                                             device_map={"": 0}, torch_dtype=torch.float16)
model.config.use_cache = True
dfs = {}
summaries = json.load(open(f"{OUT_DIR}/baseline_results.json"))
print(len(summaries), "baseline summaries loaded")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

3 baseline summaries loaded


# QLoRA fine-tune

In [ ]:

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
model = get_peft_model(model, lora)
model.print_trainable_parameters()

args = TrainingArguments(
    output_dir=f"{OUT_DIR}/qwen-resume-lora", per_device_train_batch_size=2, per_device_eval_batch_size=2,
    gradient_accumulation_steps=8, num_train_epochs = 1 if SMOKE else 2, learning_rate=2e-4, lr_scheduler_type="cosine",
    warmup_steps=1 if SMOKE else 6, fp16=True, optim="paged_adamw_8bit", logging_steps=1 if SMOKE else 10,
    eval_strategy="steps", eval_steps=2 if SMOKE else 50, save_strategy="steps", save_steps=2 if SMOKE else 50, save_total_limit=2,
    gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="none", remove_unused_columns=False)

tok.padding_side = "right"
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                  data_collator=DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100, pad_to_multiple_of=8))
model.config.use_cache = False
print("batch:", args.per_device_train_batch_size, "x accum", args.gradient_accumulation_steps)
torch.cuda.reset_peak_memory_stats()
trainer.train()
model.save_pretrained(f"{OUT_DIR}/qwen-resume-lora-final")
print("peak GPU mem since training started (GB):", round(torch.cuda.max_memory_allocated() / 1e9, 2))

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820
batch: 2 x accum 8


Step,Training Loss,Validation Loss
50,0.002805,0.003432
100,0.000622,0.002664
126,0.000491,0.002640


peak GPU mem since training started (GB): 8.01


free memory left over from the failed run

In [ ]:
# try: del trainer
# except NameError: pass
# gc.collect(); torch.cuda.empty_cache()
# model.zero_grad(set_to_none=True)

# args = TrainingArguments(
#     output_dir=f"{OUT_DIR}/qwen-resume-lora", per_device_train_batch_size=2, per_device_eval_batch_size=2,
#     gradient_accumulation_steps=8, num_train_epochs=1 if SMOKE else 2, learning_rate=2e-4, lr_scheduler_type="cosine",
#     warmup_steps=1 if SMOKE else 6, fp16=True, optim="paged_adamw_8bit", logging_steps=1 if SMOKE else 10,
#     eval_strategy="steps", eval_steps=2 if SMOKE else 50, save_strategy="steps", save_steps=2 if SMOKE else 50,
#     save_total_limit=2, gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
#     report_to="none", remove_unused_columns=False)

# tok.padding_side = "right"
# trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
#                   data_collator=DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100, pad_to_multiple_of=8))
# model.config.use_cache = False
# trainer.train()
# model.save_pretrained(f"{OUT_DIR}/qwen-resume-lora-final")

# Evaluate fine-tuned model on the SAME test sets

In [ ]:

model.config.use_cache = True
for tag, exs in (("seen formats", test_seen), ("unseen format", test_unseen), ("real records", test_real)):
    dfs[f"ft | {tag}"], s = evaluate(model, exs, f"fine-tuned | {tag}")
    summaries.append(s); print(s)

res = pd.DataFrame(summaries).set_index("set").round(3)
print(res.T)
res.to_csv(f"{OUT_DIR}/results.csv")
print("training peak (GB):", round(torch.cuda.max_memory_allocated() / 1e9, 2))

{'set': 'fine-tuned | seen formats', 'n': 100, 'json_valid': np.float64(1.0), 'schema_valid': np.float64(1.0), 'f_name': np.float64(1.0), 'f_email': np.float64(0.9895833333333334), 'f_skills': np.float64(0.9987959866220736), 'f_education': np.float64(0.99), 'f_experience': np.float64(1.0), 'field_mean': np.float64(0.9951627647714604), 'sec_per_sample': 2.691811487674713}
{'set': 'fine-tuned | unseen format', 'n': 98, 'json_valid': np.float64(1.0), 'schema_valid': np.float64(1.0), 'f_name': np.float64(1.0), 'f_email': np.float64(1.0), 'f_skills': np.float64(0.9987714149204833), 'f_education': np.float64(0.9897959183673469), 'f_experience': np.float64(1.0), 'field_mean': np.float64(0.9971048620116942), 'sec_per_sample': 2.779074773496511}
{'set': 'fine-tuned | real records', 'n': 40, 'json_valid': np.float64(1.0), 'schema_valid': np.float64(1.0), 'f_name': nan, 'f_email': nan, 'f_skills': np.float64(0.9746997300944671), 'f_education': np.float64(0.8833333333333332), 'f_experience': np.fl

# Failure analysis: worst fine-tuned outputs on the unseen format

In [ ]:

d = dfs["ft | unseen format"].sort_values("field_mean").head(3)
by_id = {e["id"]: e for e in test_unseen}
for _, r in d.iterrows():
    print("=" * 80, f"\nid {r['id']} | json_valid={r['json_valid']} schema_valid={r['schema_valid']} field_mean={r['field_mean']:.2f}")
    print("MODEL OUTPUT:", r["raw"][:500])
    print("GOLD        :", json.dumps(by_id[r["id"]]["target"], ensure_ascii=False)[:500])

id 101 | json_valid=True schema_valid=True field_mean=0.73
MODEL OUTPUT: {"name": "[redacted name]", "email": "", "skills": ["Python", "Statsmodels", "AWS", "Machine learning", "Sklearn", "Scipy"], "education": [{"institution": "UIT-RGPV", "degree": "B.E", "field": "UIT-RGPV"}], "experience": [{"company": "Matelabs", "title": "Data Scientist"}]}
GOLD        : {"name": "[redacted name]", "email": "", "skills": ["Python", "Statsmodels", "AWS", "Machine learning", "Sklearn", "Scipy", "Keras"], "education": [{"institution": "", "degree": "B.E", "field": "UIT-RGPV"}], "experience": [{"company": "Matelabs", "title": "Data Scientist"}]}
id 135 | json_valid=True schema_valid=True field_mean=0.99
MODEL OUTPUT: {"name": "", "email": "", "skills": ["C", "C++", "J2EE", "Spring", "Hibernate", "MySQL", "PostgreSQL", "Functional Testing", "Manual Testing", "Windows", "Linux"], "education": [{"institution": "RTMNU University - Jhulelal Institute of Technology", "degree": "B.E", "field": "Computer Scie

In [16]:
for k, d in dfs.items():
    name = re.sub(r"[^a-z0-9]+", "_", k.lower()).strip("_")
    d.to_json(f"{OUT_DIR}/preds_{name}.jsonl", orient="records", lines=True)
print(sorted(os.listdir(OUT_DIR)))

['baseline_results.json', 'preds_ft_real_records.jsonl', 'preds_ft_seen_formats.jsonl', 'preds_ft_unseen_format.jsonl', 'qwen-resume-lora', 'qwen-resume-lora-final', 'results.csv']


In [17]:
def skill_pr(df, exs):
    gold = {e["id"]: {norm(s) for s in e["target"]["skills"]} for e in exs}
    P, R = [], []
    for _, r in df.iterrows():
        o = parse_json(r["raw"]); g = gold[r["id"]]
        p = {norm(s) for s in (o or {}).get("skills", []) if isinstance(s, str)}
        if p: P.append(len(p & g) / len(p))
        if g: R.append(len(p & g) / len(g))
    return round(float(np.mean(P)), 3), round(float(np.mean(R)), 3)

for k, exs in (("ft | seen formats", test_seen), ("ft | unseen format", test_unseen)):
    print(k, "skills precision, recall:", skill_pr(dfs[k], exs))

ft | seen formats skills precision, recall: (1.0, 0.998)
ft | unseen format skills precision, recall: (1.0, 0.998)


In [20]:
# print(f"records over the caps: education>{MAX_EDU}: {sum(len(as_list(get(r,'education')))>MAX_EDU for r in records)}, "
#       f"experience>{MAX_EXP}: {sum(len(as_list(get(r,'experience')))>MAX_EXP for r in records)}, "
#       f"skills>{MAX_SKILLS}: {sum(len(dedupe(skill_names(get(r,'skill'))))>MAX_SKILLS for r in records)} (of {len(records)})")
# print("K.D. High in record 43's text:", "K.D. High" in next(e["text"] for e in test_unseen if e["id"] == 43))

In [19]:
# save the loss log for the README
json.dump(trainer.state.log_history, open(f"{OUT_DIR}/train_log.json", "w"), indent=2)

# worst real-record failures: model vs gold
by_id_real = {e["id"]: e for e in test_real}
for _, r in dfs["ft | real records"].sort_values("field_mean").head(5).iterrows():
    o = parse_json(r["raw"]) or {}; g = by_id_real[r["id"]]["target"]
    print("="*80, f"\nid {r['id']} | field_mean={r['field_mean']:.2f}")
    for k in ("education", "experience", "skills"):
        print(f"MODEL {k}:", o.get(k)); print(f"GOLD  {k}:", g[k])

id 173 | field_mean=0.33
MODEL education: [{'institution': '', 'degree': '', 'field': ''}]
GOLD  education: []
MODEL experience: [{'company': 'SAP Consulting', 'title': 'SAP Technical Architect'}]
GOLD  experience: [{'company': '', 'title': 'SAP Technical Architect'}]
MODEL skills: ['SAP ABAP', 'SAP NetWeaver Gateway', 'oData', 'Fiori']
GOLD  skills: ['SAP ABAP', 'SAP NetWeaver Gateway', 'oData', 'Fiori']
id 80 | field_mean=0.33
MODEL education: [{'institution': '', 'degree': '', 'field': ''}]
GOLD  education: []
MODEL experience: [{'company': 'SAP Consulting', 'title': 'SAP Technical Architect'}]
GOLD  experience: [{'company': '', 'title': 'SAP Technical Architect'}]
MODEL skills: ['SAP ABAP', 'SAP Netweaver Gateway', 'oData', 'Fiori']
GOLD  skills: ['SAP ABAP', 'SAP Netweaver Gateway', 'oData', 'Fiori']
id 75 | field_mean=0.64
MODEL education: [{'institution': 'UIT-RGPV', 'degree': 'B.E', 'field': ''}]
GOLD  education: [{'institution': '', 'degree': 'B.E', 'field': 'UIT-RGPV'}]
MODEL

In [21]:
from collections import Counter
def fp(t):
    return json.dumps({"s": sorted(norm(s) for s in t["skills"]),
                       "e": sorted((norm(x["institution"]), norm(x["degree"]), norm(x["field"])) for x in t["education"]),
                       "x": sorted((norm(x["company"]), norm(x["title"])) for x in t["experience"])})

train_fps = {fp(e["target"]) for e in train_ex + val_ex}
for name, exs in (("real", test_real), ("seen", test_seen)):
    fps = [fp(e["target"]) for e in exs]
    print(f"{name}: {len(exs)} test resumes | repeated inside the test set: {len(fps) - len(set(fps))} | identical to a train/val resume: {sum(f in train_fps for f in fps)}")

nameless = [i for i in usable if not targets[i]["name"]]
print("all usable records:", len(usable), "->", len({fp(targets[i]) for i in usable}), "unique | nameless:", len(nameless), "->", len({fp(targets[i]) for i in nameless}), "unique")

real: 40 test resumes | repeated inside the test set: 10 | identical to a train/val resume: 13
seen: 100 test resumes | repeated inside the test set: 0 | identical to a train/val resume: 2
all usable records: 4803 -> 4703 unique | nameless: 153 -> 73 unique


In [22]:
seen_fp, clean_real = set(), []
for e in test_real:
    f = fp(e["target"])
    if f in train_fps or f in seen_fp: continue      # skip leaked and repeated copies
    seen_fp.add(f); clean_real.append(e)
print("clean real test resumes:", len(clean_real), "of", len(test_real))

ids = {e["id"] for e in clean_real}
d = dfs["ft | real records"]; d = d[d["id"].isin(ids)]
fc = [c for c in d.columns if c.startswith("f_")] + ["field_mean"]
print("fine-tuned, clean real:", len(d), "| valid JSON:", d.json_valid.mean().round(3), "| schema:", d.schema_valid.mean().round(3))
print(d[fc].mean().round(3).to_dict())

with model.disable_adapter():                          # untrained base model, same 4-bit setup
    dfb, sb = evaluate(model, clean_real, "zero-shot | clean real")
print(sb)

for name, exs in (("seen", test_seen), ("unseen", test_unseen)):
    keep = {e["id"] for e in exs if fp(e["target"]) not in train_fps}
    x = dfs["ft | " + ("seen formats" if name == "seen" else "unseen format")]; x = x[x["id"].isin(keep)]
    print(f"fine-tuned {name} without leaked resumes: n={len(x)}, field score {x['field_mean'].mean():.4f}")

clean real test resumes: 22 of 40
fine-tuned, clean real: 22 | valid JSON: 1.0 | schema: 1.0
{'f_name': nan, 'f_email': nan, 'f_skills': 0.973, 'f_education': 0.879, 'f_experience': 0.955, 'field_mean': 0.935}
{'set': 'zero-shot | clean real', 'n': 22, 'json_valid': np.float64(1.0), 'schema_valid': np.float64(0.45454545454545453), 'f_name': nan, 'f_email': nan, 'f_skills': np.float64(0.7966154681829739), 'f_education': np.float64(0.6893939393939394), 'f_experience': np.float64(0.893939393939394), 'field_mean': np.float64(0.7933162671721025), 'sec_per_sample': 5.175459688360041}
fine-tuned seen without leaked resumes: n=98, field score 0.9978
fine-tuned unseen without leaked resumes: n=96, field score 0.9998


In [23]:
clean = {"n_clean_real": len(clean_real),
         "fine_tuned": d[fc].mean().round(4).to_dict(),
         "zero_shot": {k: (v if isinstance(v, str) else float(v)) for k, v in sb.items()}}
json.dump(clean, open(f"{OUT_DIR}/results_clean.json", "w"), indent=2)

In [24]:
!pip freeze | grep -i -E "^(transformers|peft|bitsandbytes|accelerate|datasets|torch)=="
!python --version

accelerate==1.15.0
bitsandbytes==0.50.2
datasets==5.0.1
peft==0.21.0
torch==2.11.0+cu128
transformers==5.17.0
Python 3.13.15


In [ ]:
RUN_BASELINE = False   # True regenerates the zero-shot baseline (~30 min); False loads baseline_results.json